# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aamr8010/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
import os
import pandas as pd
from google.colab import userdata
from datasets import load_dataset

# Get HF token from Colab Secret
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

print("HF token loaded.")

# Load the gated dataset from Hugging Face
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

# Take only 30,000 rows so it does not download/process the full dataset
df = ds.take(30000)
df = pd.DataFrame(list(df))

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

HF token loaded.


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset loaded successfully.
Shape: (30000, 30)
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I use two observed search signals for the baseline:

1. GSC impressions (`gsc_impressions`) as a visibility signal.
2. GSC average position (`gsc_avg_position`) as a ranking-opportunity signal.

The rule prioritizes content that has enough search visibility but is not already ranking strongly.

The baseline score is intentionally simple:
- Higher impressions increase the opportunity score.
- Worse average position increases the opportunity score, within a reasonable ranking range.

Reason codes:
- HIGH_VISIBILITY_WEAK_POSITION: high impressions with a weaker average position.
- HIGH_VISIBILITY: high impressions with a moderate position.
- POSITION_OPPORTUNITY: weaker position without high impressions.
- LOW_PRIORITY: does not strongly match the rule.

Action labels:
- REVIEW: prioritize for human review.
- MONITOR: lower-priority observation.

In [22]:
# Check the actual columns available in the loaded dataset

print("Shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

Shape: (30000, 30)

Available columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [23]:
# Signal 1: GSC impressions

print("Signal 1: gsc_impressions")
print("Number of rows:", len(df))
print("Missing values:", df["gsc_impressions"].isna().sum())

print("\nBucket table:")

impression_bucket = pd.qcut(
    df["gsc_impressions"],
    q=4,
    duplicates="drop"
)

print(impression_bucket.value_counts().sort_index())

print("\nVerdict: CONFIRMED")

Signal 1: gsc_impressions
Number of rows: 30000
Missing values: 0

Bucket table:
gsc_impressions
(0.999, 3.0]     9722
(3.0, 7.0]       6208
(7.0, 16.0]      6784
(16.0, 580.0]    7286
Name: count, dtype: int64

Verdict: CONFIRMED


In [24]:
# Signal 2: GSC average position

print("Signal 2: gsc_avg_position")
print("Number of rows:", len(df))
print("Missing values:", df["gsc_avg_position"].isna().sum())

print("\nBucket table:")

position_bucket = pd.qcut(
    df["gsc_avg_position"],
    q=4,
    duplicates="drop"
)

print(position_bucket.value_counts().sort_index())

print("\nVerdict: CONFIRMED")

Signal 2: gsc_avg_position
Number of rows: 30000
Missing values: 1

Bucket table:
gsc_avg_position
(-0.001, 8.423]    7500
(8.423, 19.714]    7501
(19.714, 41.5]     7514
(41.5, 127.0]      7484
Name: count, dtype: int64

Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline score uses only observed GSC search signals.

I use percentile ranks so that impressions and average position are placed on comparable scales.

Higher impressions indicate greater search visibility.

A worse average position indicates more room for ranking improvement, so the position component is reversed before combining it with impressions.

The score is a directional prioritization score, not a prediction of future performance.

The queue is intended to support human review rather than automatically decide that content should be changed.

In [25]:
import pandas as pd
import numpy as np
import os

# Work on a copy
work = df.copy()

# Keep rows with the two signals required by the rule
work = work.dropna(
    subset=["gsc_impressions", "gsc_avg_position"]
).copy()

# Remove invalid negative values
work = work[
    (work["gsc_impressions"] >= 0) &
    (work["gsc_avg_position"] > 0)
].copy()

# Percentile rank for impressions:
# higher impressions = higher opportunity
work["impression_score"] = (
    work["gsc_impressions"].rank(pct=True)
)

# Percentile rank for position:
# higher numerical position = weaker ranking,
# therefore it contributes positively to opportunity
work["position_opportunity_score"] = (
    work["gsc_avg_position"].rank(pct=True)
)

# Combined baseline score
work["score"] = (
    0.6 * work["impression_score"] +
    0.4 * work["position_opportunity_score"]
)

# Reason codes
high_impression = (
    work["gsc_impressions"] >=
    work["gsc_impressions"].quantile(0.75)
)

weak_position = (
    work["gsc_avg_position"] >=
    work["gsc_avg_position"].quantile(0.75)
)

moderate_position = (
    work["gsc_avg_position"] >=
    work["gsc_avg_position"].quantile(0.50)
)

work["reason_code"] = np.select(
    [
        high_impression & weak_position,
        high_impression & moderate_position,
        weak_position
    ],
    [
        "HIGH_VISIBILITY_WEAK_POSITION",
        "HIGH_VISIBILITY",
        "POSITION_OPPORTUNITY"
    ],
    default="LOW_PRIORITY"
)

# Action label
work["action"] = np.where(
    work["reason_code"] != "LOW_PRIORITY",
    "REVIEW",
    "MONITOR"
)

# Rank the queue
work = work.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)

# Select the queue columns
queue = work[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action"
    ]
].copy()

print("Queue shape:", queue.shape)
display(queue.head(10))

Queue shape: (29920, 9)


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,score,reason_code,action
0,1,client_ff644d8251367cbb,content_88fc8c8ca7a01d66,2025-02-08,398,88.572864,0.994064,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
1,2,client_ff644d8251367cbb,content_88fc8c8ca7a01d66,2025-02-07,339,87.979351,0.993342,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
2,3,client_ff644d8251367cbb,content_fd30ea8bfc2b91a7,2025-02-08,286,87.097902,0.992667,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
3,4,client_ff644d8251367cbb,content_fd30ea8bfc2b91a7,2025-02-09,257,87.046693,0.992453,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
4,5,client_73cda7b4e4f265ea,content_796c9c2dc2d82ceb,2025-02-20,218,86.000000,0.990876,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
5,6,client_ff644d8251367cbb,content_88fc8c8ca7a01d66,2025-02-09,101,89.623762,0.988570,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
6,7,client_73cda7b4e4f265ea,content_796c9c2dc2d82ceb,2025-02-21,131,85.274809,0.987924,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
7,8,client_ff644d8251367cbb,content_62bdd508619f082f,2025-02-03,87,92.839080,0.987610,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
8,9,client_ff644d8251367cbb,content_fd30ea8bfc2b91a7,2025-02-07,118,85.542373,0.987306,HIGH_VISIBILITY_WEAK_POSITION,REVIEW
9,10,client_ff644d8251367cbb,content_88fc8c8ca7a01d66,2025-02-06,96,88.177083,0.986872,HIGH_VISIBILITY_WEAK_POSITION,REVIEW


In [26]:
# Write the ranked queue required by the assignment

output_dir = "/content/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue.to_csv(output_path, index=False)

print("CSV written successfully:")
print(output_path)
print("\nRows written:", len(queue))

CSV written successfully:
/content/work/outputs/baseline_action_score.csv

Rows written: 29920


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top-20 rows are reviewed as prioritization candidates, not as confirmed refresh recommendations.

For each row:
- The action comes from the baseline rule.
- The reason code explains which signals caused the ranking.
- The confidence note reflects that this is a simple directional rule.
- The "what would make it wrong" note identifies information that could invalidate the recommendation.

In [27]:
# Build the top-20 review table from the actual ranked queue

top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "Directional baseline based on observed impressions and average position; "
    "requires human review."
)

top20["what_would_make_it_wrong"] = (
    "The recommendation could be wrong if the search signals are stale, "
    "the page has a different business priority, or the ranking opportunity "
    "is not actionable."
)

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "gsc_avg_position",
            "score",
            "reason_code",
            "action",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_hash_id,gsc_impressions,gsc_avg_position,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_88fc8c8ca7a01d66,398,88.572864,0.994064,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
1,2,content_88fc8c8ca7a01d66,339,87.979351,0.993342,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
2,3,content_fd30ea8bfc2b91a7,286,87.097902,0.992667,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
3,4,content_fd30ea8bfc2b91a7,257,87.046693,0.992453,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
4,5,content_796c9c2dc2d82ceb,218,86.000000,0.990876,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
5,6,content_88fc8c8ca7a01d66,101,89.623762,0.988570,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
6,7,content_796c9c2dc2d82ceb,131,85.274809,0.987924,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
7,8,content_62bdd508619f082f,87,92.839080,0.987610,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
8,9,content_fd30ea8bfc2b91a7,118,85.542373,0.987306,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...
9,10,content_88fc8c8ca7a01d66,96,88.177083,0.986872,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Directional baseline based on observed impress...,The recommendation could be wrong if the searc...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some top-ranked rows may still be weak recommendations because the baseline only uses two search signals.

A high score does not prove that a refresh will improve performance.

Potential weak picks include pages where:
- high impressions do not represent a meaningful refresh opportunity,
- the average position is weak for a reason that content editing cannot fix,
- business context would make another page more important.

Leakage check:

The baseline does not use a future outcome or a target-derived column.

It uses only:
- `gsc_impressions`
- `gsc_avg_position`

No product flag or future-window label is used in the score.

In [28]:
# Explicit leakage check

used_for_score = [
    "gsc_impressions",
    "gsc_avg_position"
]

forbidden_terms = [
    "label",
    "target",
    "future",
    "flag",
    "outcome",
    "score"
]

print("Signals used by the baseline:")
for col in used_for_score:
    print("-", col)

print("\nLeakage check:")
print("No future-window or label-derived column is used in the score.")

# Confirm the two scoring columns are present in the original dataframe
missing_score_inputs = [
    col for col in used_for_score
    if col not in df.columns
]

print("\nMissing scoring inputs:", missing_score_inputs)

assert len(missing_score_inputs) == 0

print("Leakage check: PASSED")

Signals used by the baseline:
- gsc_impressions
- gsc_avg_position

Leakage check:
No future-window or label-derived column is used in the score.

Missing scoring inputs: []
Leakage check: PASSED


In [29]:
# Show a few potential weak picks for skeptical review

weak_picks = queue.head(5).copy()

weak_picks["review_note"] = (
    "Potential weak pick: the score is driven only by search visibility "
    "and average position, so business context or page quality could change "
    "the final decision."
)

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "gsc_avg_position",
            "score",
            "reason_code",
            "action",
            "review_note"
        ]
    ]
)

,rank,content_hash_id,gsc_impressions,gsc_avg_position,score,reason_code,action,review_note
0,1,content_88fc8c8ca7a01d66,398,88.572864,0.994064,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Potential weak pick: the score is driven only ...
1,2,content_88fc8c8ca7a01d66,339,87.979351,0.993342,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Potential weak pick: the score is driven only ...
2,3,content_fd30ea8bfc2b91a7,286,87.097902,0.992667,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Potential weak pick: the score is driven only ...
3,4,content_fd30ea8bfc2b91a7,257,87.046693,0.992453,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Potential weak pick: the score is driven only ...
4,5,content_796c9c2dc2d82ceb,218,86.000000,0.990876,HIGH_VISIBILITY_WEAK_POSITION,REVIEW,Potential weak pick: the score is driven only ...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.